# Market supply / demand (nth latest savegame `.pkl`)

Per-market **demand**, **supply**, and **supply/demand** for a configurable list of goods, from a format-2 datalocations pickle.

**`NTH_LATEST` (1-based):** snapshots are sorted **newest first** (`YYYYMMDD_HHMMSS` stem). `1` = newest pkl, `2` = second-newest, etc. Folder resolution uses `resolve_pkl_dir` (current playthrough under `analysis/savegame/notebooks/save_game_temp` by default).

**`supply_demand_ratio`:** `supply / demand` when `demand > 0`; otherwise `0` (after cleanup below).

**`base_price` / `rel_price`:** `base_price` is `default_market_price` from parsed game+mod goods. **`rel_price`** is market price vs that baseline as a percent (**100** = at base, **50** = half, **200** = double). If `base_price` is 0, `rel_price` is 0.

**`net_v` / `mkt_sz`:** In EU5 the market screen **net** for a good is **`supply − demand`** (same units as those columns in the save). We use the PKL **`supply`** and **`demand`** from `market_goods` only — no extra balance field. **`net_v`** is that per row; **`net_mkt`** sums **`net_v`** over your selected goods per market (mixed units if you mix goods). **`mkt_px`** is still Σ `(demand+supply)×price` for “priced activity” size. **`mkt_sz`** also has one column per selected good with that market's **`net_v`** (0 if the market–good row was dropped as no-flow). See **`mkt_sz`** below.

**Cleanup:** (1) Drop `market_goods` rows where **both** flows are **below 0.01** (NaNs treated as 0). (2) Build the full market × goods grid and merge; absent pairs get **0**. (3) Drop grid rows where **both** `supply` and `demand` stay **below 0.01** so empty market–good combinations do not appear.

**Modding hint:** ratio **> 1** → local glut; **< 1** → shortage (candidates for RGO/building tweaks).

**Goods list:** The first code cell can set `GOODS` manually or, with `LOAD_GOODS_FROM_GAME = True`, from `GoodsSubsets` (`GOODS_SUBSET_ATTR` e.g. `rgo` / `raw_material`, `farming`, `all`, `vanilla_food`).

In [10]:
from pathlib import Path

import numpy as np
import pandas as pd

from analysis.building_levels.building_analysis.utils import load_config
from analysis.savegame.processor import resolve_pkl_dir
from core.data.goods_data import GoodsData
from core.data.goods_subsets import GoodsSubsets
from core.parser.path_resolver import PathResolver

# --- config ---
# Override pickle directory (must exist), or None for default playthrough folder.
PKL_DIR: Path | None = None

# 1-based index, newest first (1 = latest snapshot).
NTH_LATEST = 1

# Good list: either set manually, or load from parsed game+mod via GoodsSubsets.
LOAD_GOODS_FROM_GAME = False
# When True: attribute on GoodsSubsets, e.g. "rgo"/"raw_material", "all", "farming", "mining", "vanilla_food", …
GOODS_SUBSET_ATTR = "vanilla_food"

# Used only when LOAD_GOODS_FROM_GAME is False — good slugs as in save `good_id`.
GOODS_MANUAL: list[str] = ["wheat", "victuals"]

if LOAD_GOODS_FROM_GAME:
    _cfg = load_config()
    _gd = GoodsData(PathResolver(_cfg["game_path"], _cfg["mod_path"]))
    _gd.load_all()
    _gs = GoodsSubsets.from_goods_data(_gd)
    _subset = getattr(_gs, GOODS_SUBSET_ATTR, None)
    if _subset is None:
        raise ValueError(
            f"Unknown GOODS_SUBSET_ATTR={GOODS_SUBSET_ATTR!r} on GoodsSubsets."
        )
    if not isinstance(_subset, tuple):
        raise ValueError(
            f"GOODS_SUBSET_ATTR must be a tuple field (e.g. rgo, raw_material, all, farming), "
            f"not {type(_subset).__name__}."
        )
    GOODS: list[str] = list(_subset)
else:
    GOODS = list(GOODS_MANUAL)
    _cfg = load_config()
    _gd = GoodsData(PathResolver(_cfg["game_path"], _cfg["mod_path"]))
    _gd.load_all()

print(f"GOODS ({len(GOODS)} selected):")
print(GOODS)

# Optional: set to a path to write CSV; None skips export.
EXPORT_PATH: Path | None = None

GOODS (2 selected):
['wheat', 'victuals']


In [11]:
base = resolve_pkl_dir(PKL_DIR)
pkls = sorted(base.glob("*.pkl"), key=lambda p: p.stem, reverse=True)
if not pkls:
    raise FileNotFoundError(f"No .pkl files in {base}")
if not (1 <= NTH_LATEST <= len(pkls)):
    raise ValueError(
        f"NTH_LATEST={NTH_LATEST} out of range: {len(pkls)} pkls in {base}"
    )

pkl_path = pkls[NTH_LATEST - 1]
print(f"Using ({NTH_LATEST}/{len(pkls)} newest): {pkl_path.name}")
print(f"Directory: {base}")

payload = pd.read_pickle(pkl_path)
if payload.get("format") != 2:
    raise ValueError(f"Expected format 2 payload, got {payload.get('format')!r}")
if "market_goods" not in payload:
    raise KeyError("market_goods missing from payload")

mg = payload["market_goods"].copy()
print(f"market_goods rows: {len(mg):,}")

Using (1/1 newest): 20260411_152919.pkl
Directory: C:\Development\ProsperPerishCalcs\analysis\savegame\notebooks\save_game_temp\2f340d96_fa76_4031_a994_a995bc772dbe
market_goods rows: 9,750


In [12]:
if not GOODS:
    raise ValueError("GOODS list is empty")

required = {"market_id", "market_center_slug", "good_id", "supply", "demand"}
missing_cols = required - set(mg.columns)
if missing_cols:
    raise ValueError(f"market_goods missing columns: {sorted(missing_cols)}")

MIN_SUPPLY_DEMAND = 0.01  # drop market_goods rows where both flows are below this
_mg_n = len(mg)
_sup = pd.to_numeric(mg["supply"], errors="coerce")
_dem = pd.to_numeric(mg["demand"], errors="coerce")
_both_tiny = (_sup.fillna(0) < MIN_SUPPLY_DEMAND) & (_dem.fillna(0) < MIN_SUPPLY_DEMAND)
mg = mg.loc[~_both_tiny].copy()
print(
    f"market_goods: dropped {_mg_n - len(mg):,} rows with supply and demand both < {MIN_SUPPLY_DEMAND}; "
    f"{len(mg):,} rows remain"
)

markets = mg[["market_id", "market_center_slug"]].drop_duplicates()
market_ids = markets.sort_values("market_id")["market_id"].unique()

grid = pd.MultiIndex.from_product(
    [market_ids, GOODS],
    names=["market_id", "good_id"],
).to_frame(index=False)
grid = grid.merge(markets, on="market_id", how="left")

merge_cols = ["market_id", "good_id", "supply", "demand"]
if "price" in mg.columns:
    merge_cols.append("price")

sub = mg.loc[mg["good_id"].isin(GOODS), merge_cols].copy()
result = grid.merge(sub, on=["market_id", "good_id"], how="left")

result["demand"] = pd.to_numeric(result["demand"], errors="coerce").fillna(0.0)
result["supply"] = pd.to_numeric(result["supply"], errors="coerce").fillna(0.0)
if "price" not in result.columns:
    result["price"] = 0.0
result["price"] = pd.to_numeric(result["price"], errors="coerce").fillna(0.0)

# Game UI "net" column: supply − demand (goods units), not × price.
result["net_v"] = result["supply"] - result["demand"]

d = result["demand"].to_numpy(dtype=float, copy=False)
s = result["supply"].to_numpy(dtype=float, copy=False)
_sd_ratio = np.zeros_like(s, dtype=np.float64)
np.divide(s, d, out=_sd_ratio, where=d > 0)
result["supply_demand_ratio"] = _sd_ratio

_round_cols = ["demand", "supply", "net_v", "price", "supply_demand_ratio"]
result[_round_cols] = result[_round_cols].round(2)

# Drop grid rows with no save data: left-merge fills missing (market, good) with 0.
_result_n = len(result)
result = result.loc[
    ~(
        (result["supply"] < MIN_SUPPLY_DEMAND)
        & (result["demand"] < MIN_SUPPLY_DEMAND)
    )
].copy()
print(
    f"result: dropped {_result_n - len(result):,} no-flow rows (both supply & demand < {MIN_SUPPLY_DEMAND}); "
    f"{len(result):,} rows remain"
)

_base_map = pd.to_numeric(_gd.modded_df["default_market_price"], errors="coerce")
result["base_price"] = (
    result["good_id"].astype(str).map(_base_map).fillna(0.0).astype(float)
)
_bp = result["base_price"].to_numpy(dtype=float, copy=False)
_mp = result["price"].to_numpy(dtype=float, copy=False)
result["rel_price"] = np.where(_bp > 0, 100.0 * _mp / _bp, 0.0)
result[["base_price", "rel_price"]] = result[["base_price", "rel_price"]].round(2)

# Per market: sum over selected goods of demand*price + supply*price (same as (demand+supply)*price).
_px_row = (result["demand"] + result["supply"]) * result["price"]
result["mkt_px"] = (
    _px_row.groupby(result["market_id"], sort=False).transform("sum").round(2)
)
result["mkt_rnk"] = (
    result["mkt_px"].rank(method="dense", ascending=False).astype(int)
)
result = result.sort_values(
    ["mkt_rnk", "market_id", "good_id"],
    ascending=[True, True, True],
).reset_index(drop=True)

# Σ (supply − demand) over selected goods (same units as in-game net per good; summing mixes goods).
_net_mkt = (
    result["supply"] - result["demand"]
).groupby(result["market_id"], sort=False).sum().round(2).rename("net_mkt")
mkt_sz = (
    result.drop_duplicates(subset=["market_id"])[
        ["mkt_rnk", "market_id", "mkt_px", "market_center_slug"]
    ]
    .merge(_net_mkt.reset_index(), on="market_id")
    .sort_values("mkt_rnk", kind="mergesort")
    .reset_index(drop=True)
)

# One column per selected good: net_v (supply − demand); missing = no-flow → 0.
_net_wide = result.pivot(index="market_id", columns="good_id", values="net_v")
_net_wide = _net_wide.reindex(columns=list(GOODS)).fillna(0.0).round(2)
_net_wide.columns.name = None
mkt_sz = mkt_sz.merge(_net_wide.reset_index(), on="market_id")

display_cols = [
    "market_id",
    "mkt_rnk",
    "mkt_px",
    "market_center_slug",
    "good_id",
    "demand",
    "supply",
    "net_v",
    "price",
    "base_price",
    "rel_price",
    "supply_demand_ratio",
]

print(f"Grid: {len(result):,} rows ({len(market_ids)} markets × {len(GOODS)} goods)")

market_goods: dropped 3,750 rows with supply and demand both < 0.01; 6,000 rows remain
result: dropped 19 no-flow rows (both supply & demand < 0.01); 241 rows remain
Grid: 241 rows (130 markets × 2 goods)


### Markets by size (`mkt_sz`)

One row per **market**: **`mkt_rnk`**, **`mkt_px`**, **`net_mkt`** (Σ **`net_v`** over selected goods), then **one column per good** in your filter with **`net_v`** = `supply−demand` (same as the in-game net column). **`mkt_sz.head(20)`**: top 20 markets by **`mkt_px`**.

In [13]:
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 200)

_mkt_show = [
    "mkt_rnk",
    "market_id",
    "mkt_px",
    "net_mkt",
    "market_center_slug",
    *GOODS,
]
mkt_sz.head(20)[_mkt_show]

,mkt_rnk,market_id,mkt_px,net_mkt,market_center_slug,wheat,victuals
0,1,90,616.74,-54.80,delhi,-7.28,-47.52
1,2,92,494.74,-39.07,varanasi,0.00,-39.08
2,3,30,449.66,-18.19,nuremberg,0.00,-18.19
3,4,11,398.88,74.01,paris,50.32,23.70
4,5,67,352.26,-5.60,hangzhou,-2.36,-3.23
5,6,14,348.85,-13.42,cologne,0.00,-13.42
6,7,75,311.21,22.59,nanchang,9.74,12.85
7,8,70,309.29,122.16,kaifeng,106.82,15.34
8,9,4,307.72,-10.78,genoa,3.94,-14.72
9,10,71,300.77,54.98,shangyuan,56.63,-1.65


In [14]:
pd.options.display.max_columns = None
pd.options.display.precision = 2

# Shortages first (low ratio), within each good
by_scarcity = result[display_cols].sort_values(
    ["good_id", "supply_demand_ratio"],
    ascending=[True, True],
    na_position="last",
)
by_scarcity

,market_id,mkt_rnk,mkt_px,market_center_slug,good_id,demand,supply,net_v,price,base_price,rel_price,supply_demand_ratio
222,63,117,1.08,urgench,victuals,0.00,0.83,0.83,1.02,2.5,40.8,0.00
226,62,119,0.86,sarayjuk,victuals,0.00,0.28,0.28,0.92,2.5,36.8,0.00
228,124,120,0.49,male_atoll,victuals,0.00,0.38,0.38,1.30,2.5,52.0,0.00
229,120,121,0.37,chimgi_tura,victuals,0.00,0.10,0.10,1.44,2.5,57.6,0.00
235,125,124,0.20,balj,victuals,0.00,0.07,0.07,1.48,2.5,59.2,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...
90,33,47,53.07,alexandria,wheat,9.33,38.96,29.63,0.27,1.0,27.0,4.18
162,122,84,19.40,tarnovo,wheat,2.62,11.10,8.48,0.64,1.0,64.0,4.24
191,58,100,9.96,al_ahsa,wheat,0.14,0.74,0.60,0.64,1.0,64.0,5.15
212,39,111,4.97,algiers,wheat,0.49,3.37,2.88,0.62,1.0,62.0,6.84


In [15]:
# Gluts: high ratio first
by_glut = result[display_cols].sort_values(
    ["good_id", "supply_demand_ratio"],
    ascending=[True, False],
    na_position="last",
)
by_glut

,market_id,mkt_rnk,mkt_px,market_center_slug,good_id,demand,supply,net_v,price,base_price,rel_price,supply_demand_ratio
224,128,118,1.02,kurchum,victuals,0.00,0.65,0.65,1.26,2.5,50.4,64710.00
237,117,125,0.14,cahokia,victuals,0.00,0.09,0.09,1.52,2.5,60.8,2895.33
239,98,127,0.02,ialophaybo,victuals,0.00,0.01,0.01,2.29,2.5,91.6,306.75
233,127,123,0.30,boorj,victuals,0.03,0.18,0.15,1.21,2.5,48.4,6.68
20,65,11,288.10,kyoto,victuals,35.82,92.72,56.91,2.19,2.5,87.6,2.59
...,...,...,...,...,...,...,...,...,...,...,...,...
225,128,118,1.02,kurchum,wheat,0.12,0.00,-0.12,1.64,1.0,164.0,0.00
230,120,121,0.37,chimgi_tura,wheat,0.13,0.00,-0.13,1.74,1.0,174.0,0.00
232,83,122,0.34,karakorum,wheat,0.07,0.00,-0.07,1.89,1.0,189.0,0.00
234,127,123,0.30,boorj,wheat,0.03,0.00,-0.03,1.47,1.0,147.0,0.00


In [16]:
# Per-good summary (finite ratio only for min/median/max)
def _ratio_stats(r: pd.Series) -> pd.Series:
    finite = r[np.isfinite(r)]
    return pd.Series(
        {
            "n_markets": len(r),
            "n_finite_ratio": int(finite.size),
            "ratio_median": finite.median() if finite.size else np.nan,
            "ratio_min": finite.min() if finite.size else np.nan,
            "ratio_max": finite.max() if finite.size else np.nan,
        }
    )


result.groupby("good_id", sort=True)["supply_demand_ratio"].apply(_ratio_stats)

good_id                 
victuals  n_markets           128.00
          n_finite_ratio      128.00
          ratio_median          0.81
          ratio_min             0.00
          ratio_max         64710.00
wheat     n_markets           113.00
          n_finite_ratio      113.00
          ratio_median          1.00
          ratio_min             0.00
          ratio_max             8.01
Name: supply_demand_ratio, dtype: float64

### Drill-down: full `market_goods` row for one (market, good)

Use this when you need `supplied_*` / `demanded_*` breakdown columns from the save.

In [17]:
# Example: set to a market_id / good_id from the grid above
DRILL_MARKET_ID = None # e.g. 1
DRILL_GOOD = None  # e.g. "iron"

if DRILL_MARKET_ID is not None and DRILL_GOOD:
    row = mg[(mg["market_id"] == DRILL_MARKET_ID) & (mg["good_id"] == DRILL_GOOD)]
    if row.empty:
        print("No row in market_goods for that pair.")
    else:
        s = row.iloc[0]
        non_null = s[s.notna() & (s.astype(str) != "")]
        display(non_null)
else:
    print("Set DRILL_MARKET_ID and DRILL_GOOD to inspect flattened save fields.")

Set DRILL_MARKET_ID and DRILL_GOOD to inspect flattened save fields.


In [18]:
if EXPORT_PATH is not None:
    out = Path(EXPORT_PATH)
    out.parent.mkdir(parents=True, exist_ok=True)
    result[display_cols].to_csv(out, index=False)
    print(f"Wrote {out}")